## 1. Project Introduction

### 1.1 Problem Statement

Bike-sharing systems generate large amounts of usage data containing information
about time, weather, seasonality, and working-day conditions.

The goal of this project is to develop regression models that can predict the
total number of bike rentals in a given hour.

### 1.2 Machine Learning Problem

This is a **supervised regression problem** because:

- The input variables describe the conditions for a particular hour.
- The target variable `cnt` represents the total number of bike rentals.
- The target is a continuous numerical value.

### 1.3 Target Variable

The target variable is:

`cnt` — Total number of bike rentals.

### 1.4 Project Goal

The main goal is to compare multiple regression algorithms and identify a model
that provides strong predictive performance while maintaining reliable
generalisation on unseen test data.

## 2. Import Libraries

The libraries required for data manipulation, visualization, preprocessing,
model training, evaluation, hyperparameter tuning, and dimensionality reduction
are imported below.

In [2]:

import numpy as np
import pandas as pd

import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split, GridSearchCV, cross_val_score
from sklearn.preprocessing import StandardScaler, OneHotEncoder, PolynomialFeatures
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer

from sklearn.linear_model import LinearRegression, Ridge, Lasso, ElasticNet
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.svm import SVR
from sklearn.neighbors import KNeighborsRegressor

from sklearn.decomposition import PCA

from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error

import warnings
warnings.filterwarnings("ignore")

RANDOM_STATE = 42

print("All required libraries imported successfully.")

All required libraries imported successfully.


## 3. Loading the Dataset

The hourly Bike Sharing dataset is loaded into a Pandas DataFrame.

The dataset contains information about weather, season, calendar information, time of day, and bike rental counts.

In [4]:
# Load the hourly Bike Sharing dataset
df = pd.read_csv("../Data/hour.csv")

# Display the first five rows
df.head()


,instant,dteday,season,yr,mnth,hr,holiday,weekday,workingday,weathersit,temp,atemp,hum,windspeed,casual,registered,cnt
0,1,2011-01-01,1,0,1,0,0,6,0,1,0.24,0.2879,0.81,0.0,3,13,16
1,2,2011-01-01,1,0,1,1,0,6,0,1,0.22,0.2727,0.80,0.0,8,32,40
2,3,2011-01-01,1,0,1,2,0,6,0,1,0.22,0.2727,0.80,0.0,5,27,32
3,4,2011-01-01,1,0,1,3,0,6,0,1,0.24,0.2879,0.75,0.0,3,10,13
4,5,2011-01-01,1,0,1,4,0,6,0,1,0.24,0.2879,0.75,0.0,0,1,1


## 4. Initial Data Understanding

Before preprocessing the dataset, we inspect its structure.

The following aspects are examined:

- Number of rows and columns
- Column names
- Data types
- Basic statistical information
- Target variable distribution

In [6]:
# Dataset dimensions
print("Dataset shape:", df.shape)
# Display column names
print("Columns:")
print(df.columns.tolist())

Dataset shape: (17379, 17)
Columns:
['instant', 'dteday', 'season', 'yr', 'mnth', 'hr', 'holiday', 'weekday', 'workingday', 'weathersit', 'temp', 'atemp', 'hum', 'windspeed', 'casual', 'registered', 'cnt']


In [7]:
# Display data types and non-null counts
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 17379 entries, 0 to 17378
Data columns (total 17 columns):
 #   Column      Non-Null Count  Dtype  
---  ------      --------------  -----  
 0   instant     17379 non-null  int64  
 1   dteday      17379 non-null  object 
 2   season      17379 non-null  int64  
 3   yr          17379 non-null  int64  
 4   mnth        17379 non-null  int64  
 5   hr          17379 non-null  int64  
 6   holiday     17379 non-null  int64  
 7   weekday     17379 non-null  int64  
 8   workingday  17379 non-null  int64  
 9   weathersit  17379 non-null  int64  
 10  temp        17379 non-null  float64
 11  atemp       17379 non-null  float64
 12  hum         17379 non-null  float64
 13  windspeed   17379 non-null  float64
 14  casual      17379 non-null  int64  
 15  registered  17379 non-null  int64  
 16  cnt         17379 non-null  int64  
dtypes: float64(4), int64(12), object(1)
memory usage: 2.3+ MB


In [8]:
# Statistical summary of numerical columns
df.describe().T

,count,mean,std,min,25%,50%,75%,max
instant,17379.0,8690.000000,5017.029500,1.00,4345.5000,8690.0000,13034.5000,17379.0000
season,17379.0,2.501640,1.106918,1.00,2.0000,3.0000,3.0000,4.0000
yr,17379.0,0.502561,0.500008,0.00,0.0000,1.0000,1.0000,1.0000
mnth,17379.0,6.537775,3.438776,1.00,4.0000,7.0000,10.0000,12.0000
hr,17379.0,11.546752,6.914405,0.00,6.0000,12.0000,18.0000,23.0000
holiday,17379.0,0.028770,0.167165,0.00,0.0000,0.0000,0.0000,1.0000
weekday,17379.0,3.003683,2.005771,0.00,1.0000,3.0000,5.0000,6.0000
workingday,17379.0,0.682721,0.465431,0.00,0.0000,1.0000,1.0000,1.0000
weathersit,17379.0,1.425283,0.639357,1.00,1.0000,1.0000,2.0000,4.0000
temp,17379.0,0.496987,0.192556,0.02,0.3400,0.5000,0.6600,1.0000


In [9]:
# Target variable overview
print("Target variable: cnt")
print("Minimum:", df["cnt"].min())
print("Maximum:", df["cnt"].max())
print("Mean:", df["cnt"].mean())
print("Median:", df["cnt"].median())

Target variable: cnt
Minimum: 1
Maximum: 977
Mean: 189.46308763450142
Median: 142.0


## 5. Data Quality Analysis

Data quality is checked before model development.

We examine:

- Missing values
- Duplicate rows
- Potentially invalid values
- Data types
- Duplicate treatment

These checks help ensure that the dataset is suitable for subsequent analysis and modelling.